# 03a – Baseline Regression Models
**Owner:** Member 3 (Classical Machine Learning Engineer)

Establishes minimum performance benchmarks using `DummyRegressor(strategy="mean")` and compares it with `LinearRegression` inside the shared preprocessing pipeline.

In [1]:
import sys
import time
from pathlib import Path

import joblib
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CV_FOLDS, MODELS_DIR
from src.evaluate import (
    compute_metrics,
    cross_validate_model,
    plot_parity,
    plot_residuals,
    save_model_run_metadata,
    save_predictions,
)
from src.features import build_preprocessor, get_xy

MODELS_DIR.mkdir(parents=True, exist_ok=True)
X_train, y_train, id_train = get_xy("train")
X_val, y_val, id_val = get_xy("val")
X_test, y_test, id_test = get_xy("test")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (3499, 193), Val: (750, 193), Test: (750, 193)


In [2]:
# 1. Primary Baseline: DummyRegressor (mean)
dummy_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", DummyRegressor(strategy="mean")),
])

cv_dummy = cross_validate_model(dummy_pipe, X_train, y_train, cv=CV_FOLDS)
print("DummyRegressor 5-Fold CV:", {k: round(v, 4) for k, v in cv_dummy.items()})

t0 = time.perf_counter()
dummy_pipe.fit(X_train, y_train)
dummy_time = time.perf_counter() - t0

val_pred_dummy = dummy_pipe.predict(X_val)
test_pred_dummy = dummy_pipe.predict(X_test)

print("Dummy Val Metrics :", compute_metrics(y_val, val_pred_dummy))
print("Dummy Test Metrics:", compute_metrics(y_test, test_pred_dummy))

joblib.dump(dummy_pipe, MODELS_DIR / "baseline_dummy.joblib")
save_predictions(id_val, y_val, val_pred_dummy, "baseline_dummy", "val")
save_predictions(id_test, y_test, test_pred_dummy, "baseline_dummy", "test")
save_model_run_metadata(
    model_name="baseline_dummy",
    display_name="Baseline (Dummy Mean)",
    training_time_sec=dummy_time,
    cv_summary=cv_dummy,
)

DummyRegressor 5-Fold CV: {'r2_mean': -0.0006, 'r2_std': 0.0006, 'rmse_mean': 1.5313, 'rmse_std': 0.0338, 'mae_mean': 1.1981, 'mae_std': 0.0302}
Dummy Val Metrics : {'r2': -0.00888983100299745, 'rmse': 1.685981934958013, 'mae': 1.288261921975513}
Dummy Test Metrics: {'r2': -0.0007072645636829744, 'rmse': 1.4636196043868603, 'mae': 1.168147011955488}


WindowsPath('C:/Users/nourh/.gemini/antigravity/scratch/mof-co2-uptake-prediction/results/baseline_dummy_meta.json')

In [3]:
# 2. Optional Linear Benchmark: Ordinary Least Squares LinearRegression
lin_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", LinearRegression()),
])

cv_lin = cross_validate_model(lin_pipe, X_train, y_train, cv=CV_FOLDS)
print("LinearRegression 5-Fold CV:", {k: round(v, 4) for k, v in cv_lin.items()})

t0 = time.perf_counter()
lin_pipe.fit(X_train, y_train)
lin_time = time.perf_counter() - t0

val_pred_lin = lin_pipe.predict(X_val)
test_pred_lin = lin_pipe.predict(X_test)

print("LinearRegression Val Metrics :", compute_metrics(y_val, val_pred_lin))
print("LinearRegression Test Metrics:", compute_metrics(y_test, test_pred_lin))

joblib.dump(lin_pipe, MODELS_DIR / "baseline_linear.joblib")
save_predictions(id_val, y_val, val_pred_lin, "baseline_linear", "val")
save_predictions(id_test, y_test, test_pred_lin, "baseline_linear", "test")
save_model_run_metadata(
    model_name="baseline_linear",
    display_name="Linear Regression",
    training_time_sec=lin_time,
    cv_summary=cv_lin,
)

plot_parity(y_test, test_pred_lin, "baseline_linear", split="test")
plot_residuals(y_test, test_pred_lin, "baseline_linear", split="test")
print("Saved baseline models, predictions, and diagnostic plots.")

LinearRegression 5-Fold CV: {'r2_mean': -16.2824, 'r2_std': 32.4951, 'rmse_mean': 4.0522, 'rmse_std': 4.9993, 'mae_mean': 1.1476, 'mae_std': 0.2322}
LinearRegression Val Metrics : {'r2': -122.6948647933662, 'rmse': 18.668390626901047, 'mae': 1.7481440328603042}
LinearRegression Test Metrics: {'r2': -0.007584318033599047, 'rmse': 1.4686401318550055, 'mae': 0.9845988906075207}


Saved baseline models, predictions, and diagnostic plots.
